# Sketch2Shape AI — TripoSR on Colab GPU
Run cells in order in a **GPU runtime**. This notebook clones official TripoSR and runs its pretrained inference directly: no paid API, tunnel, server or Gradio UI. The explicit inference cell downloads public weights on first use. No generation happens on import.

Upload this notebook to Colab, select **Runtime → Change runtime type → T4 GPU**, and supply a ZIP of your project (include untracked backend files; omit environments, `.git`, secrets and outputs), or use a Drive folder. GPU availability is not guaranteed. This notebook's GPU inference has not been tested by the local offline tests.

In [ ]:
import os, sys, subprocess, platform, shutil, re
from pathlib import Path

def run_checked(command, **kwargs):
    try:
        return subprocess.run(command, check=True, **kwargs)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError) as exc:
        raise RuntimeError(f"Setup failed: {command[0]}. Read the command output above; stop here and fix the environment before continuing. {exc}") from exc

import torch
print("Python:", sys.version, "Platform:", platform.platform())
print("PyTorch:", torch.__version__, "PyTorch CUDA:", torch.version.cuda)
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError("This dependency recipe targets Python 3.10–3.12. Select a compatible Colab runtime before continuing.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU unavailable. Select a GPU runtime and rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
run_checked(["nvidia-smi"])
if not shutil.which("nvcc"):
    raise RuntimeError("CUDA toolkit/nvcc is required to compile torchmcubes.")
nvcc = subprocess.check_output(["nvcc", "--version"], text=True)
print(nvcc)
match = re.search(r"release (\d+)\.", nvcc)
if not match or not torch.version.cuda or match.group(1) != torch.version.cuda.split('.')[0]:
    raise RuntimeError("CUDA toolkit and PyTorch CUDA major versions must match. Use a compatible runtime; do not blindly replace torch.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0] Platform: Linux-6.6.159+-x86_64-with-glibc2.35
PyTorch: 2.11.0+cu128 PyTorch CUDA: 12.8
GPU: Tesla T4
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0



## Obtain your project source — choose ZIP or Drive (no credentials in code)

In [ ]:
SOURCE_MODE = "zip"  # or "drive"
DRIVE_PROJECT = "/content/drive/MyDrive/Sketch2Shape"  # edit if using Drive

if SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path(DRIVE_PROJECT).resolve()
elif SOURCE_MODE == "zip":
    from google.colab import files
    import zipfile, io, stat
    print("Upload one project ZIP, excluding .venv, .git, weights, outputs and secrets.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one ZIP.")
    archive = next(iter(uploaded.values()))
    source_root = Path("/content/sketch2shape-source")
    if source_root.exists():
        raise FileExistsError("Source already exists. Reuse PROJECT or reset runtime; this cell will not overwrite it.")
    source_root.mkdir()
    with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
        if sum(item.file_size for item in bundle.infolist()) > 512 * 1024**2:
            raise ValueError("Source ZIP exceeds 512 MiB unpacked; exclude environments and outputs.")
        for item in bundle.infolist():
            target = (source_root / item.filename).resolve()
            if not target.is_relative_to(source_root.resolve()) or stat.S_ISLNK(item.external_attr >> 16):
                raise ValueError("ZIP contains an unsafe path or symlink.")
        bundle.extractall(source_root)
    candidates = [p.parent.parent for p in source_root.rglob("backend/generator.py")]
    if len(candidates) != 1:
        raise ValueError("ZIP must contain exactly one project with backend/generator.py.")
    PROJECT = candidates[0].resolve()
    del archive, uploaded
else:
    raise ValueError("SOURCE_MODE must be zip or drive")
if not (PROJECT / "backend/generator.py").is_file():
    raise FileNotFoundError("Project backend not found")
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Project:", PROJECT)

Upload one project ZIP, excluding .venv, .git, weights, outputs and secrets.


Saving sketch2shape_colab.zip to sketch2shape_colab (2).zip


ValueError: ZIP must contain exactly one project with backend/generator.py.

In [ ]:
from pathlib import Path
import zipfile

zips = sorted(
    Path("/content").glob("sketch2shape_colab*.zip"),
    key=lambda p: p.stat().st_mtime
)

zip_path = zips[-1]
print("Checking:", zip_path.name)

with zipfile.ZipFile(zip_path) as z:
    for name in z.namelist():
        if "generator.py" in name or "backend" in name:
            print(repr(name))

Checking: sketch2shape_colab (2).zip
'backend\\.gitignore'
'backend\\generator.py'
'backend\\make_example.py'
'backend\\README.md'
'backend\\requirements.txt'
'backend\\validate_mesh.py'
'backend\\__init__.py'
'backend\\tests\\test_generator.py'
'backend\\tests\\__init__.py'
'backend\\tests\\__pycache__\\test_generator.cpython-312.pyc'
'backend\\tests\\__pycache__\\__init__.cpython-312.pyc'
'backend\\__pycache__\\generator.cpython-312.pyc'
'backend\\__pycache__\\make_example.cpython-312.pyc'
'backend\\__pycache__\\validate_mesh.cpython-312.pyc'
'backend\\__pycache__\\__init__.cpython-312.pyc'


In [ ]:
from pathlib import Path
import zipfile
import shutil
import os
import sys

target = Path("/content/sketch2shape-fixed")
target.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path) as z:
    for member in z.infolist():
        name = member.filename.replace("\\", "/")
        parts = Path(name).parts

        if not name or ".." in parts or name.startswith("/"):
            continue

        destination = target.joinpath(*parts)

        if member.is_dir() or name.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
            continue

        destination.parent.mkdir(parents=True, exist_ok=True)

        with z.open(member) as source, open(destination, "wb") as output:
            shutil.copyfileobj(source, output)

matches = list(target.rglob("backend/generator.py"))

if len(matches) != 1:
    raise RuntimeError(
        f"Expected one backend/generator.py, found {len(matches)}. "
        f"Matches: {matches}"
    )

PROJECT = matches[0].parent.parent.resolve()
os.chdir(PROJECT)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("Project ready:", PROJECT)
print("Backend exists:", (PROJECT / "backend/generator.py").is_file())

Project ready: /content/sketch2shape-fixed
Backend exists: True


In [ ]:
from pathlib import Path

PROJECT = Path("/content/sketch2shape-source")

print("Folder exists:", PROJECT.exists())
print("Backend exists:", (PROJECT / "backend" / "generator.py").exists())

Folder exists: True
Backend exists: False


In [ ]:
import shutil
from pathlib import Path

folder = Path("/content/sketch2shape-source")

if folder.exists():
    shutil.rmtree(folder)
    print("Old extraction removed successfully.")
else:
    print("Folder does not exist.")

Folder does not exist.


In [ ]:
import torch
from pathlib import Path

print("Project:", PROJECT)
print("Backend found:", (PROJECT / "backend/generator.py").is_file())
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project: /content/sketch2shape-fixed
Backend found: True
GPU available: True
GPU: Tesla T4


In [ ]:
import sys
import torch
import shutil
import importlib.util
import subprocess

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("nvcc location:", shutil.which("nvcc"))

for package in ["scikit_build_core", "pybind11"]:
    print(package, "installed:", importlib.util.find_spec(package) is not None)

if shutil.which("nvcc"):
    result = subprocess.run(
        ["nvcc", "--version"],
        capture_output=True, text=True
    )
    print(result.stdout)

Python: 3.12.13
PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
nvcc location: /usr/local/cuda/bin/nvcc
scikit_build_core installed: False
pybind11 installed: False
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0



In [ ]:
%pip install -U scikit-build-core pybind11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.4/319.4 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 8.8 MB/s eta 0:00:00


In [ ]:
%pip install --no-build-isolation -v git+https://github.com/tatsy/torchmcubes.git

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-khdocvam
  Running command git version
  git version 2.34.1
  Running command git clone --filter=blob:none https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-khdocvam
  Cloning into '/tmp/pip-req-build-khdocvam'...
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Resolved https://github.com/tatsy/torchmcubes.git to commit a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command git rev-parse HEAD
  a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Running command Preparing metadata (pyproject.toml)
  2026-10-31 08:34:44,731 - scikit_build_core - INFO - RUN: /usr/local/lib/python3.12/dist-packages/cmake/data/bin/cmake -E capabilities
  2026-10-31 08:34:44,939 - scikit_build_core - INFO - CMake version: 3.31.10
  *** scikit-build-core 1.1.1 using CMake 3.31.10 (metadata_wheel)
  2026-10-31 08:34:44,93

In [ ]:
from pathlib import Path
import os
import sys

matches = list(Path("/content/sketch2shape-fixed").rglob("backend/generator.py"))

assert len(matches) == 1, f"Expected one backend, found {len(matches)}"

PROJECT = matches[0].parent.parent.resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

print("Project restored:", PROJECT)

Project restored: /content/sketch2shape-fixed


In [ ]:
from torchmcubes import marching_cubes
print("torchmcubes is working!")

torchmcubes is working!


In [ ]:
from pathlib import Path
import os
import sys

matches = list(Path("/content/sketch2shape-fixed").rglob("backend/generator.py"))
assert len(matches) == 1, f"Expected one backend, found {len(matches)}"

PROJECT = matches[0].parent.parent.resolve()

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

print("Project restored:", PROJECT)
print("Backend exists:", (PROJECT / "backend" / "generator.py").is_file())

Project restored: /content/sketch2shape-fixed
Backend exists: True


## Clone upstream and install inference dependencies
This modifies only the disposable Colab environment, keeping its preinstalled PyTorch. Upstream's 2024 pins are adapted for Python 3.10–3.12 (notably tokenizers/Pillow/NumPy compatibility). Gradio is unnecessary. Native torchmcubes must compile against the current torch and CUDA toolkit; installation may take several minutes. Failures stop the workflow. Do not continue after an error; inspect pip/compiler output and select a compatible runtime. The cloned commit is printed for reproducibility.

In [1]:
from pathlib import Path
import importlib.util
import os

# Find the cloned TripoSR repository
matches = [
    p.parent for p in Path("/content").rglob("run.py")
    if (p.parent / "tsr").is_dir()
]

print("TripoSR repositories found:", matches)

for package in ["torch", "torchmcubes", "trimesh", "PIL", "transformers", "rembg"]:
    print(package, ":", "Installed" if importlib.util.find_spec(package) else "Missing")

if matches:
    TRIPOSR_DIR = str(matches[0].resolve())
    os.environ["TRIPOSR_DIR"] = TRIPOSR_DIR
    print("TRIPOSR_DIR:", TRIPOSR_DIR)

TripoSR repositories found: []
torch : Installed
torchmcubes : Missing
trimesh : Missing
PIL : Installed
transformers : Installed
rembg : Missing


In [2]:
from pathlib import Path
import os
import subprocess

repo = Path("/content/TripoSR")

if not (repo / "run.py").is_file():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/VAST-AI-Research/TripoSR.git",
        str(repo)
    ], check=True)

os.environ["TRIPOSR_DIR"] = str(repo)
print("TripoSR repository ready:", repo)

TripoSR repository ready: /content/TripoSR


In [3]:
%pip install scikit-build-core pybind11
%pip install omegaconf==2.3.0 einops trimesh "rembg[cpu]" "imageio[ffmpeg]" huggingface-hub xatlas moderngl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.4/293.4 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.4/319.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 10.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.5/751.5 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.4/305.4 kB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 120.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 123.

In [1]:
%pip install --no-build-isolation git+https://github.com/tatsy/torchmcubes.git

  Cloning https://github.com/tatsy/torchmcubes.git to /tmp/pip-req-build-3m4n2i9l
  Running command git clone --filter=blob:none --quiet https://github.com/tatsy/torchmcubes.git /tmp/pip-req-build-3m4n2i9l
  Resolved https://github.com/tatsy/torchmcubes.git to commit a9ac95aa329957ba646e7bb057f825f5dfa77f38
  Preparing metadata (pyproject.toml) ... done
  Created wheel for torchmcubes: filename=torchmcubes-0.1.0-cp312-cp312-linux_x86_64.whl size=159740 sha256=b8987b1ec5e14b3ebd62d16a72953539e7b1443ff833564ecdcb0d7fbcee5bd5
  Stored in directory: /tmp/pip-ephem-wheel-cache-ex02aavz/wheels/7c/f7/fb/1cc460d15449f41b86a30b1ee3e693970bdd177c0c02a2c053
Successfully built torchmcubes


In [2]:
from pathlib import Path
import importlib.util
import os

repo = Path("/content/TripoSR")

print("TripoSR:", (repo / "run.py").is_file())

for pkg in ["torchmcubes", "trimesh", "rembg", "omegaconf", "einops", "xatlas", "moderngl"]:
    print(pkg, bool(importlib.util.find_spec(pkg)))

os.environ["TRIPOSR_DIR"] = str(repo)

from torchmcubes import marching_cubes
print("Basic dependency check passed")

TripoSR: True
torchmcubes True
trimesh True
rembg True
omegaconf True
einops True
xatlas True
moderngl True
Basic dependency check passed


In [4]:
from pathlib import Path

matches = [
    p for p in Path("/content").rglob("generator.py")
    if p.parent.name == "backend"
]

print("Backend files found:", len(matches))

for p in matches:
    print(p)

Backend files found: 0


In [5]:
from pathlib import Path
from google.colab import files
import zipfile
import tempfile
import shutil
import os
import sys

# Find the previously uploaded ZIP
zips = sorted(
    Path("/content").glob("sketch2shape_colab*.zip"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if zips:
    zip_path = zips[0]
    print("Found ZIP:", zip_path)
else:
    print("Please upload sketch2shape_colab.zip")
    uploaded = files.upload()
    zip_path = Path(next(iter(uploaded)))

# Extract into a fresh directory
target = Path(tempfile.mkdtemp(
    prefix="sketch2shape-",
    dir="/content"
))

with zipfile.ZipFile(zip_path) as archive:
    for item in archive.infolist():
        name = item.filename.replace("\\", "/")
        parts = [p for p in name.split("/") if p and p != "."]

        if name.startswith("/") or ".." in parts or any(":" in p for p in parts):
            raise ValueError("Unsafe ZIP path detected")

        if not parts:
            continue

        destination = target.joinpath(*parts)

        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as src, open(destination, "wb") as dst:
                shutil.copyfileobj(src, dst)

# Locate the backend
matches = [
    p for p in target.rglob("generator.py")
    if p.parent.name == "backend"
]

print("Backend files found:", len(matches))

assert len(matches) == 1, "Backend not found in extracted ZIP"

PROJECT = matches[0].parent.parent.resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

os.environ["TRIPOSR_DIR"] = "/content/TripoSR"

print("Project restored:", PROJECT)
print("Backend ready:", (PROJECT / "backend/generator.py").is_file())

Please upload sketch2shape_colab.zip


Saving sketch2shape_colab.zip to sketch2shape_colab.zip
Backend files found: 1
Project restored: /content/sketch2shape-a2dr44uq
Backend ready: True


In [6]:
import torch
from pathlib import Path

print("Backend:", (PROJECT / "backend/generator.py").is_file())
print("TripoSR:", Path("/content/TripoSR/run.py").is_file())
print("GPU:", torch.cuda.is_available())

Backend: True
TripoSR: True
GPU: True


In [8]:
import subprocess
import sys
from pathlib import Path

repo = Path("/content/TripoSR")
image = PROJECT / "examples/chair.png"
output = Path("/content/triposr-debug")

assert (repo / "run.py").is_file(), "TripoSR not found"
assert image.is_file(), "Input image not found"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        str(repo / "run.py"),
        str(image),
        "--output-dir", str(output),
        "--model-save-format", "glb",
    ],
    cwd=str(repo),
    check=False
)

print("Exit code:", result.returncode)

if result.returncode == 0:
    print("Generated files:", list(output.rglob("*.glb")))
else:
    print("TripoSR failed. Check the error above.")

Exit code: 1
TripoSR failed. Check the error above.


In [9]:
import subprocess
import sys
from pathlib import Path

repo = Path("/content/TripoSR")
image = PROJECT / "examples/chair.png"
output = Path("/content/triposr-debug")

result = subprocess.run(
    [
        sys.executable,
        "-u",
        str(repo / "run.py"),
        str(image),
        "--output-dir", str(output),
        "--model-save-format", "glb",
    ],
    cwd=str(repo),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
    check=False
)

# Save complete diagnostic output
log_file = Path("/content/triposr-error.log")
log_file.write_text(result.stdout or "", encoding="utf-8")

print("Exit code:", result.returncode)
print("\n===== LAST 80 LOG LINES =====\n")
print("\n".join((result.stdout or "").splitlines()[-80:]))
print("\nFull log saved to:", log_file)

if result.returncode == 0:
    print("Generated GLBs:", list(output.rglob("*.glb")))

Exit code: 1

===== LAST 80 LOG LINES =====

2026-10-31 09:51:57,327 - INFO - Initializing model ...
2026-10-31 09:51:57,727 - INFO - HTTP Request: HEAD https://huggingface.co/stabilityai/TripoSR/resolve/main/config.yaml "HTTP/1.1 307 Temporary Redirect"
2026-10-31 09:51:57,728 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-31 09:51:57,738 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/stabilityai/TripoSR/5b521936b01fbe1890f6f9baed0254ab6351c04a/config.yaml "HTTP/1.1 200 OK"
2026-10-31 09:51:57,825 - INFO - HTTP Request: HEAD https://huggingface.co/stabilityai/TripoSR/resolve/main/model.ckpt "HTTP/1.1 302 Found"
2026-10-31 09:52:07,352 - INFO - NumExpr defaulting to 2 threads.
2026-10-31 09:52:08,505 - INFO - HTTP Request: HEAD https://huggingface.co/facebook/dino-vitb16/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-31 09:52:08

In [10]:
import transformers

print("Current Transformers version:", transformers.__version__)

Current Transformers version: 5.13.1


In [11]:
%pip install --force-reinstall --no-deps "transformers==4.40.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 111.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [12]:
%pip install "tokenizers==0.19.1" "huggingface-hub==0.25.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 58.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.39.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.25.2 which is incompatible.
gradio 6.20.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is i

In [1]:
import transformers
import tokenizers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Transformers: 4.40.2
Tokenizers: 0.19.1
Hugging Face Hub: 0.25.2


In [2]:
from pathlib import Path
import os
import sys
import torch

matches = [
    p for p in Path("/content").rglob("generator.py")
    if p.parent.name == "backend"
]

assert len(matches) == 1, f"Found {len(matches)} backend files"

PROJECT = matches[0].parent.parent.resolve()
TRIPOSR_DIR = "/content/TripoSR"

os.chdir(PROJECT)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

os.environ["TRIPOSR_DIR"] = TRIPOSR_DIR

print("Backend:", (PROJECT / "backend/generator.py").is_file())
print("TripoSR:", Path(TRIPOSR_DIR, "run.py").is_file())
print("Image:", (PROJECT / "examples/chair.png").is_file())
print("GPU:", torch.cuda.is_available())

Backend: True
TripoSR: True
Image: True
GPU: True


In [3]:
from backend.generator import generate_3d
import time

start = time.time()

model_path = generate_3d(
    str(PROJECT / "examples/chair.png")
)

print("Generated 3D model:", model_path)
print("Time taken:", round(time.time() - start, 2), "seconds")

GenerationError: TripoSR exited with code -9; see upstream console logs for dependency, CUDA, weight-download or inference errors

In [4]:
!free -h
!nvidia-smi

               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.2Gi       8.8Gi       5.0Mi       665Mi       9.2Gi
Swap:             0B          0B          0B
Sat Oct 31 10:02:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8        

In [5]:
from pathlib import Path

p = Path("/sys/fs/cgroup/memory.events")
if p.exists():
    print(p.read_text())

low 0
high 0
max 0
oom 0
oom_kill 0
oom_group_kill 0



In [6]:
import subprocess
import sys
from pathlib import Path

repo = Path("/content/TripoSR")
image = PROJECT / "examples/chair.png"
output = Path("/content/triposr-lowmem")

command = [
    sys.executable,
    "-u",
    str(repo / "run.py"),
    str(image),
    "--output-dir", str(output),
    "--model-save-format", "glb",
    "--mc-resolution", "128",
    "--chunk-size", "1024",
    "--no-remove-bg"
]

result = subprocess.run(
    command,
    cwd=str(repo),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace"
)

print("Exit code:", result.returncode)
print("\n".join(result.stdout.splitlines()[-35:]))

if result.returncode == 0:
    print("GLB files:", list(output.rglob("*.glb")))

Exit code: 1
2026-10-31 10:04:54,257 - INFO - Initializing model ...
2026-10-31 10:05:09,448 - INFO - Initializing model finished in 15191.55ms.
2026-10-31 10:05:09,449 - INFO - Processing images ...
2026-10-31 10:05:09,461 - INFO - Processing images finished in 12.60ms.
2026-10-31 10:05:09,461 - INFO - Running image 1/1 ...
2026-10-31 10:05:09,462 - INFO - Running model ...
2026-10-31 10:05:12,118 - INFO - Running model finished in 2656.70ms.
2026-10-31 10:05:12,118 - INFO - Extracting mesh ...
2026-10-31 10:05:14,218 - INFO - Extracting mesh finished in 2099.10ms.
2026-10-31 10:05:14,218 - INFO - Exporting mesh ...
Traceback (most recent call last):
  File "/content/TripoSR/run.py", line 190, in <module>
    meshes[0].export(out_mesh_path)
  File "/usr/local/lib/python3.12/dist-packages/trimesh/base.py", line 3042, in export
    return export_mesh(mesh=self, file_obj=file_obj, file_type=file_type, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [7]:
import subprocess
import sys
from pathlib import Path

repo = Path("/content/TripoSR")
image = PROJECT / "examples/chair.png"
output = Path("/content/triposr-lowmem")

cmd = [
    sys.executable,
    "-u",
    str(repo / "run.py"),
    str(image),
    "--output-dir", str(output),
    "--model-save-format", "glb",
    "--mc-resolution", "128",
    "--chunk-size", "1024",
    "--no-remove-bg"
]

result = subprocess.run(
    cmd,
    cwd=str(repo),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace"
)

print("Exit code:", result.returncode)
print("\n".join(result.stdout.splitlines()[-40:]))

if result.returncode == 0:
    print("Generated models:", list(output.rglob("*.glb")))

Exit code: 1
2026-10-31 10:05:40,868 - INFO - Initializing model ...
2026-10-31 10:05:48,308 - INFO - Initializing model finished in 7440.06ms.
2026-10-31 10:05:48,309 - INFO - Processing images ...
2026-10-31 10:05:48,315 - INFO - Processing images finished in 6.67ms.
2026-10-31 10:05:48,315 - INFO - Running image 1/1 ...
2026-10-31 10:05:48,315 - INFO - Running model ...
2026-10-31 10:05:49,415 - INFO - Running model finished in 1099.93ms.
2026-10-31 10:05:49,416 - INFO - Extracting mesh ...
2026-10-31 10:05:51,164 - INFO - Extracting mesh finished in 1748.82ms.
2026-10-31 10:05:51,165 - INFO - Exporting mesh ...
Traceback (most recent call last):
  File "/content/TripoSR/run.py", line 190, in <module>
    meshes[0].export(out_mesh_path)
  File "/usr/local/lib/python3.12/dist-packages/trimesh/base.py", line 3042, in export
    return export_mesh(mesh=self, file_obj=file_obj, file_type=file_type, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [8]:
from pathlib import Path

Path("/content/triposr-lowmem/0").mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory created!")

Output directory created!


In [9]:
import subprocess
import sys
from pathlib import Path

# Paths
repo = Path("/content/TripoSR")
image = PROJECT / "examples/chair.png"
output = Path("/content/triposr-lowmem")

# Fix the missing output directory
(output / "0").mkdir(parents=True, exist_ok=True)

# Run TripoSR
cmd = [
    sys.executable,
    "-u",
    str(repo / "run.py"),
    str(image),
    "--output-dir", str(output),
    "--model-save-format", "glb",
    "--mc-resolution", "128",
    "--chunk-size", "1024",
    "--no-remove-bg"
]

print("Starting TripoSR inference...")

result = subprocess.run(
    cmd,
    cwd=str(repo),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace"
)

print("Exit code:", result.returncode)
print("\n===== INFERENCE LOG =====\n")
print("\n".join(result.stdout.splitlines()[-40:]))

# Check generated GLB
if result.returncode == 0:
    models = list(output.rglob("*.glb"))
    print("\nGenerated GLB files:", models)
else:
    print("\nTripoSR failed. Check the error log above.")

Starting TripoSR inference...
Exit code: 0

===== INFERENCE LOG =====

2026-10-31 10:08:29,201 - INFO - Initializing model ...
2026-10-31 10:08:35,999 - INFO - Initializing model finished in 6798.21ms.
2026-10-31 10:08:36,000 - INFO - Processing images ...
2026-10-31 10:08:36,006 - INFO - Processing images finished in 6.63ms.
2026-10-31 10:08:36,006 - INFO - Running image 1/1 ...
2026-10-31 10:08:36,007 - INFO - Running model ...
2026-10-31 10:08:37,100 - INFO - Running model finished in 1093.38ms.
2026-10-31 10:08:37,100 - INFO - Extracting mesh ...
2026-10-31 10:08:39,279 - INFO - Extracting mesh finished in 2178.65ms.
2026-10-31 10:08:39,279 - INFO - Exporting mesh ...
2026-10-31 10:08:39,285 - INFO - Exporting mesh finished in 5.71ms.

Generated GLB files: [PosixPath('/content/triposr-lowmem/0/mesh.glb')]


In [10]:
from pathlib import Path
import trimesh
from google.colab import files

glb = Path("/content/triposr-lowmem/0/mesh.glb")

assert glb.is_file(), "GLB was not generated"

scene = trimesh.load(str(glb), force="scene")

meshes = [
    mesh for mesh in scene.geometry.values()
    if isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0
]

assert meshes, "GLB contains no valid mesh geometry"

print("3D model validated successfully!")
print("Vertices:", sum(len(mesh.vertices) for mesh in meshes))
print("Faces:", sum(len(mesh.faces) for mesh in meshes))

files.download(str(glb))

3D model validated successfully!
Vertices: 16915
Faces: 33716


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from backend.generator import generate_3d

model_path = generate_3d(
    str(PROJECT / "examples/chair.png")
)

print("Generated GLB:", model_path)

GenerationError: TripoSR exited with code 1; see upstream console logs for dependency, CUDA, weight-download or inference errors

In [3]:
from pathlib import Path
import os
import sys
import torch

matches = list(
    Path("/content/sketch2shape-fixed").rglob("backend/generator.py")
)

assert len(matches) == 1, f"Found {len(matches)} backend files"

PROJECT = matches[0].parent.parent.resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

TRIPOSR_DIR = "/content/TripoSR"
os.environ["TRIPOSR_DIR"] = TRIPOSR_DIR

print("Project:", PROJECT)
print("TripoSR:", TRIPOSR_DIR)
print("GPU:", torch.cuda.get_device_name(0))
print("Example image:", (PROJECT / "examples/chair.png").exists())

AssertionError: Found 0 backend files

In [ ]:
TRIPOSR = Path("/content/TripoSR")
if not TRIPOSR.exists():
    run_checked(["git", "clone", "https://github.com/VAST-AI-Research/TripoSR.git", str(TRIPOSR)], timeout=300)
else:
    origin = subprocess.check_output(["git", "-C", str(TRIPOSR), "remote", "get-url", "origin"], text=True).strip()
    if origin != "https://github.com/VAST-AI-Research/TripoSR.git":
        raise RuntimeError("Existing /content/TripoSR is not the expected upstream checkout")
print("Upstream commit:", subprocess.check_output(["git", "-C", str(TRIPOSR), "rev-parse", "HEAD"], text=True).strip())
run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "setuptools", "wheel", "ninja", "packaging"], timeout=600)
# Keep the notebook kernel's torch/CUDA build; do not install a second torch.
run_checked([sys.executable, "-m", "pip", "install",
    "numpy>=1.26,<2", "Pillow>=10.4,<11", "trimesh>=4.4,<5",
    "omegaconf==2.3.0", "einops==0.7.0", "transformers==4.44.2",
    "huggingface-hub>=0.23.2,<1", "rembg[cpu]==2.0.67",
    "imageio[ffmpeg]", "xatlas>=0.0.9,<1", "moderngl==5.10.0"], timeout=900)
os.environ["CUDA_HOME"] = str(Path(shutil.which("nvcc")).resolve().parents[1])
os.environ["TORCH_CUDA_ARCH_LIST"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
os.environ["MAX_JOBS"] = "2"
run_checked([sys.executable, "-m", "pip", "install", "--no-build-isolation",
             "git+https://github.com/tatsy/torchmcubes.git"], timeout=1200)
os.environ["TRIPOSR_DIR"] = str(TRIPOSR)
# Check imports in a fresh subprocess, not the notebook's possibly cached modules.
run_checked([sys.executable, "-c", "import torch, numpy, transformers, rembg, xatlas, moderngl; from tsr.system import TSR; import torchmcubes_module as m; assert torch.cuda.is_available(); assert hasattr(m, 'mcubes_cuda'), 'torchmcubes lacks CUDA support'; print('GPU dependencies OK', torch.__version__, numpy.__version__, transformers.__version__)"], cwd=str(TRIPOSR), timeout=120)
run_checked([sys.executable, str(TRIPOSR / "run.py"), "--help"], cwd=str(TRIPOSR), timeout=120)
print("TRIPOSR_DIR:", os.environ["TRIPOSR_DIR"])
print("If pip warned about already-imported packages, restart the session (not disconnect/delete), then rerun the setup cells. Dependencies remain installed.")

Upstream commit: 107cefdc244c39106fa830359024f6a2f1c78871


RuntimeError: Setup failed: /usr/bin/python3. Read the command output above; stop here and fix the environment before continuing. Command '['/usr/bin/python3', '-m', 'pip', 'install', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git']' returned non-zero exit status 2.

## Offline backend checks — no model weights or inference

In [ ]:
run_checked([sys.executable, "-m", "compileall", "-q", "backend"], cwd=str(PROJECT), timeout=60)
run_checked([sys.executable, "-m", "unittest", "discover", "-s", "backend/tests", "-v"], cwd=str(PROJECT), timeout=120)
# Running checks in subprocesses avoids reusing stale NumPy/Pillow modules after pip installation.

## Choose a fresh input image (or the included sketch)

In [ ]:
USE_UPLOADED_IMAGE = True
if USE_UPLOADED_IMAGE:
    from google.colab import files
    uploaded_images = files.upload()
    if len(uploaded_images) != 1:
        raise ValueError("Upload exactly one PNG/JPEG/WebP image.")
    import uuid
    input_dir = PROJECT / "examples"
    input_dir.mkdir(exist_ok=True)
    name, data = next(iter(uploaded_images.items()))
    IMAGE = input_dir / f"uploaded_{uuid.uuid4().hex}{Path(name).suffix.lower()}"
    IMAGE.write_bytes(data)
    del uploaded_images
else:
    IMAGE = PROJECT / "examples/chair.png"
if not IMAGE.is_file():
    raise FileNotFoundError(IMAGE)
print("Input:", IMAGE)
from IPython.display import display, Image as DisplayImage
display(DisplayImage(filename=str(IMAGE)))

## Explicit real inference — downloads public pretrained weights on first use
Run this cell when ready. It calls `generate_3d()` unchanged in a fresh Python process using this runtime's interpreter (avoids stale imported packages). Upstream handles model/config download from `stabilityai/TripoSR` and background-removal weights. No API key is needed. First-run downloads and extraction can take minutes; backend timeout is 30 minutes. Only a real GLB that passes validation is reported. No fallback is created.

In [ ]:
import time, json, uuid
result_record = PROJECT / "outputs" / f"notebook_result_{uuid.uuid4().hex}.json"
started = time.perf_counter()
inference_code = """
import json, logging, sys, time
from pathlib import Path
from backend.generator import generate_3d
from backend.validate_mesh import validate_mesh
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
start = time.perf_counter()
path = generate_3d(sys.argv[1], prompt=None)
info = validate_mesh(path)
assert Path(path).is_file() and Path(path).stat().st_size > 0
Path(sys.argv[2]).write_text(json.dumps({'path': path, 'mesh': info, 'seconds': time.perf_counter() - start}))
"""
run_checked([sys.executable, "-c", inference_code, str(IMAGE), str(result_record)], cwd=str(PROJECT), timeout=1860)
result = json.loads(result_record.read_text())
GLB = Path(result["path"])
if not GLB.is_file() or GLB.stat().st_size == 0:
    raise RuntimeError("Validated GLB is no longer present")
print("Validated GLB:", GLB)
print("Mesh information:", result["mesh"])
print("Inference + validation seconds:", round(result["seconds"], 2))
print("Total cell seconds:", round(time.perf_counter() - started, 2))
print("Bytes:", GLB.stat().st_size)

## Download the validated GLB — click explicitly; save before the runtime resets

In [ ]:
import ipywidgets as widgets
from google.colab import files
button = widgets.Button(description="Download validated GLB", layout=widgets.Layout(width="240px"))
status = widgets.Output()
def download_clicked(_):
    with status:
        if not GLB.is_file():
            raise FileNotFoundError("Run inference successfully before downloading")
        # Revalidate the file immediately before offering the download.
        run_checked([sys.executable, "-c", "import sys; from backend.validate_mesh import validate_mesh; print(validate_mesh(sys.argv[1]))", str(GLB)], cwd=str(PROJECT), timeout=120)
        files.download(str(GLB))
button.on_click(download_clicked)
display(button, status)

### Attribution and limits
[Official TripoSR](https://github.com/VAST-AI-Research/TripoSR) by Tripo AI / Stability AI; [public pretrained weights](https://huggingface.co/stabilityai/TripoSR), MIT license. Single-image reconstruction can be ambiguous, especially for sparse sketches. Geometry validation does not prove resemblance. Upstream defaults need about 6 GB VRAM; free Colab GPUs and session lifetime are limited. This is manual notebook generation and file download: the local frontend cannot directly access a Colab filesystem or execute here. Frontend transport remains future work. No public endpoint is created.